In [1]:
import os
import sys
from pathlib import Path
from IPython.display import Markdown, display
from rich import print
from openai import OpenAI
from rich import print
from dotenv import load_dotenv
from scraper import fetch_website_contents

src_path = Path.cwd().parent
if src_path.exists() and str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
print(f"Added to sys.path: {src_path}")

from bootstrap import OpenAIModelAPI


openai = OpenAIModelAPI.api()

Added to sys.path: /Users/jtgv/Documents/code/forks/llm_engineering

# Day 4

## Tokenizing with code

In [17]:
import tiktoken

encoding = tiktoken.encoding_for_model("gpt-4o-mini")
tokens = encoding.encode("Hi my name is Joao and I like banoffee pie")

In [18]:
tokens

[12194, 922, 1308, 382, 12501, 3458, 326, 357, 1299, 9171, 26458, 5148]

In [19]:
for token_id in tokens:
    token_text = encoding.decode([token_id])
    print(f"{token_id} = {token_text}")

12194 = Hi

922 =  my

1308 =  name

382 =  is

12501 =  Jo

...

357 =  I

1299 =  like

9171 =  ban

26458 = offee

5148 =  pie

In [5]:
encoding.decode([326])

' and'

# And another topic!

### The Illusion of "memory"

Many of you will know this already. But for those that don't -- this might be an "AHA" moment!

### You should be very comfortable with what the next cell is doing!

_I'm creating a new instance of the OpenAI Python Client library, a lightweight wrapper around making HTTP calls to an endpoint for calling the GPT LLM, or other LLM providers_

### A message to OpenAI is a list of dicts

In [8]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi! I'm Ed!"}
    ]

In [9]:
response = openai.chat.completions.create(model=OpenAIModelAPI.model(), messages=messages)
response.choices[0].message.content

'Hello, Ed! How can I assist you today?'

### OK let's now ask a follow-up question

In [10]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What's my name?"}
    ]

In [11]:
response = openai.chat.completions.create(model=OpenAIModelAPI.model(), messages=messages)
response.choices[0].message.content

"I'm sorry, but I don't have access to your personal information, including your name. How can I assist you today?"

### Wait, wha??

We just told you!

What's going on??

Here's the thing: every call to an LLM is completely STATELESS. It's a totally new call, every single time. As AI engineers, it's OUR JOB to devise techniques to give the impression that the LLM has a "memory".

In [20]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi! I'm Tiago!"},
    {"role": "assistant", "content": "Hi Tiago! How can I assist you today?"},
    {"role": "user", "content": "What's my name?"}
    ]

In [21]:
response = openai.chat.completions.create(model=OpenAIModelAPI.model(), messages=messages)
response.choices[0].message.content

'Your name is Tiago! How can I help you today?'

## To recap

With apologies if this is obvious to you - but it's still good to reinforce:

1. Every call to an LLM is stateless
2. We pass in the entire conversation so far in the input prompt, every time
3. This gives the illusion that the LLM has memory - it apparently keeps the context of the conversation
4. But this is a trick; it's a by-product of providing the entire conversation, every time
5. An LLM just predicts the most likely next tokens in the sequence; if that sequence contains "My name is Ed" and later "What's my name?" then it will predict.. Ed!

The ChatGPT product uses exactly this trick - every time you send a message, it's the entire conversation that gets passed in.

"Does that mean we have to pay extra each time for all the conversation so far"

For sure it does. And that's what we WANT. We want the LLM to predict the next tokens in the sequence, looking back on the entire conversation. We want that compute to happen, so we need to pay the electricity bill for it!

